# Camada Bronze - Ingestão de dados filtrados

## Objetivo
- Ler dados previamente filtrados no formato .txt
- Atribuir nomes corretos às colunas (EEG, biosinais e sensores)
- Padronizar estrutura dos dados entre pacientes
- Ajustar ordem de canais dependentes do paciente
- Adicionar metadados (patient_id, task_id, session)
- Converter tipos para otimização (float32)
- Reorganizar colunas em formato consistente
- Salvar os dados em formato Parquet de forma particionada

## Entrada
- data/raw/filtered_txt/

## Saída
- data/bronze/filtered_parquet/

## Observação
- Os dados já passaram por pré-processamento (filtragem e segmentação por tarefa)
- Esta etapa foca na padronização estrutural e organização
- Cada arquivo .txt é convertido individualmente em .parquet
- Pipeline otimizado para datasets grandes e uso em Machine Learning

In [10]:
import pandas as pd
import numpy as np
import os

In [11]:
def txt_to_parquet(
    txt_path,
    output_path,
    patient_id,
    task_id,
    session=None
):
    # =========================
    # DEFINIÇÃO DAS COLUNAS
    # =========================

    eeg_tags = [
        'EEG-FP1',
        'EEG-FP2',
        'EEG-F3',
        'EEG-F4',
        'EEG-C3',
        'EEG-C4',
        'EEG-P3',
        'EEG-P4',
        'EEG-O1',
        'EEG-O2',
        'EEG-F7',
        'EEG-F8',
        'EEG-P7',
        'EEG-P8',
        'EEG-FZ',
        'EEG-CZ',
        'EEG-PZ',
        'EEG-FC1',
        'EEG-FC2',
        'EEG-CP1',
        'EEG-CP2',
        'EEG-FC5',
        'EEG-FC6',
        'EEG-CP5',
        'EEG-CP6'
    ]

    # patient dependent channel order
    bio_tags = {
        "001": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
        "002": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
        "003": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
        "004": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
        "005": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
        "006": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
        "007": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
        "008": {
            "OFF_1": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
            "OFF_2": ["EMG-RTA", "EMG-RGS", "IO", "ECG", "EMG-LTA"],
        },
        "009": ["EMG-LTA", "EMG-RTA", "IO", "EMG-RGS", "ECG"],
        "010": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
        "011": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
        "012": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"]
    }

    fixed_bio_tags = ["EMG-RTA", "EMG-LTA", "EMG-RGS", "IO", "ECG"]

    acc_feats = [
        "ACCX", "ACCY", "ACCZ", "GYRO-X", "GYRO-Y", "GYRO-Z", "NC/SC"
    ]

    acc_sensors = [
        "LShank-", "RShank-", "Waist-", "Arm-"
    ]

    acc_tags = [sensor+feat for sensor in acc_sensors for feat in acc_feats]

    if session:
        patient_bio_tags = bio_tags[patient_id][session]
    else:
        patient_bio_tags = bio_tags[patient_id]
        
    columns = (
        ["timestamp"] +
        eeg_tags +
        patient_bio_tags +  
        acc_tags +
        ["label"]
    )

    final_columns = (
        ["timestamp", "patient_id", "task_id", "session"] +
        eeg_tags +
        fixed_bio_tags +  
        acc_tags +
        ["label"]
    )

    # =========================
    # LEITURA
    # =========================
    
    df = pd.read_csv(txt_path, header=None, names=columns, index_col=None)

    # =========================
    # TIPAGEM
    # =========================
    
    for col in df.columns:
        if col != "timestamp":
            df[col] = df[col].astype(np.float32)

    # =========================
    # ADICIONAR METADADOS
    # =========================
    
    df["patient_id"] = patient_id
    df["task_id"] = task_id

    if session:
        df["session"] = session
    else:
        df["session"] = pd.NA

    # =========================
    # SALVAR PARQUET
    # =========================
    
    df = df[final_columns]
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    df.to_parquet(
        output_path,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

    print(f"Arquivo salvo em: {output_path}")
    print(df.info())



In [12]:
BRONZE_PATH = "../data/raw/filtered_txt/Filtered Data/"
OUTPUT_PATH = "../data/bronze/filtered_parquet/"


def process_dataset():
    for patient in sorted(os.listdir(BRONZE_PATH)):
        patient_path = os.path.join(BRONZE_PATH, patient)

        if not os.path.isdir(patient_path):
            continue

        # =========================
        # CASO ESPECIAL: paciente 008
        # =========================
        if patient == "008":
            for session in ["OFF_1", "OFF_2"]:
                session_path = os.path.join(patient_path, session)

                if not os.path.exists(session_path):
                    continue

                for file in sorted(os.listdir(session_path)):
                    if not file.endswith(".txt"):
                        continue

                    txt_path = os.path.join(session_path, file)

                    task_id = file.replace(".txt", "")

                    output_path = os.path.join(
                        OUTPUT_PATH,
                        patient,
                        session,
                        f"{task_id}.parquet"
                    )

                    txt_to_parquet(
                        txt_path=txt_path,
                        output_path=output_path,
                        patient_id=patient,
                        task_id=task_id,
                        session=session
                    )

        # =========================
        # PACIENTES NORMAIS
        # =========================
        else:
            for file in sorted(os.listdir(patient_path)):
                if not file.endswith(".txt"):
                    continue

                txt_path = os.path.join(patient_path, file)

                task_id = file.replace(".txt", "")

                output_path = os.path.join(
                    OUTPUT_PATH,
                    patient,
                    f"{task_id}.parquet"
                )

                txt_to_parquet(
                    txt_path=txt_path,
                    output_path=output_path,
                    patient_id=patient,
                    task_id=task_id
                )

In [13]:
process_dataset()

Arquivo salvo em: ../data/bronze/filtered_parquet/001/task_1.parquet
<class 'pandas.DataFrame'>
RangeIndex: 180501 entries, 0 to 180500
Data columns (total 63 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   timestamp      180501 non-null  str    
 1   patient_id     180501 non-null  str    
 2   task_id        180501 non-null  str    
 3   session        0 non-null       object 
 4   EEG-FP1        180501 non-null  float32
 5   EEG-FP2        180501 non-null  float32
 6   EEG-F3         180501 non-null  float32
 7   EEG-F4         180501 non-null  float32
 8   EEG-C3         180501 non-null  float32
 9   EEG-C4         180501 non-null  float32
 10  EEG-P3         180501 non-null  float32
 11  EEG-P4         180501 non-null  float32
 12  EEG-O1         180501 non-null  float32
 13  EEG-O2         180501 non-null  float32
 14  EEG-F7         180501 non-null  float32
 15  EEG-F8         180501 non-null  float32
 16  EEG-P7         1

In [14]:

PARQUET_PATH = "../data/bronze/filtered_parquet/"
OUTPUT_PATH = PARQUET_PATH 

ZERO_EPS = 1e-8
MISSING_THRESHOLD = 0.95

acc_tags = [
    'LShank-ACCX', 'LShank-ACCY', 'LShank-ACCZ', 'LShank-GYRO-X',
    'LShank-GYRO-Y', 'LShank-GYRO-Z', 'LShank-NC/SC',
    'RShank-ACCX', 'RShank-ACCY', 'RShank-ACCZ', 'RShank-GYRO-X',
    'RShank-GYRO-Y', 'RShank-GYRO-Z', 'RShank-NC/SC',
    'Waist-ACCX', 'Waist-ACCY', 'Waist-ACCZ', 'Waist-GYRO-X',
    'Waist-GYRO-Y', 'Waist-GYRO-Z', 'Waist-NC/SC',
    'Arm-ACCX', 'Arm-ACCY', 'Arm-ACCZ', 'Arm-GYRO-X',
    'Arm-GYRO-Y', 'Arm-GYRO-Z', 'Arm-NC/SC'
]


def is_missing_channel(col_values):
    zero_ratio = np.mean(np.abs(col_values) < ZERO_EPS)
    # pro thiago: adicionei essa verificação por variância pq não necessariamente o valor ser menor que o ZERO_EPS significa que ele seja inexistente, principalmente para as medições do EEG
    variance = np.var(col_values)
    return (zero_ratio > 0.95) or (variance < 1e-10)


def process_parquet(file_path, output_path):
    df = pd.read_parquet(file_path)

    for col in acc_tags:
        if col not in df.columns:
            continue

        values = df[col].values

        # detecta sensor ausente
        if is_missing_channel(values):
            df[col] = np.nan

    # cria pasta destino
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    df.to_parquet(output_path, index=False)


def process_missing_sensors():
    for root, _, files in os.walk(PARQUET_PATH):
        for file in files:
            if not file.endswith(".parquet"):
                continue

            input_file = os.path.join(root, file)

            # espelha estrutura de pastas
            relative_path = os.path.relpath(input_file, PARQUET_PATH)
            output_file = os.path.join(OUTPUT_PATH, relative_path)

            print(f"Processando: {input_file}")

            process_parquet(input_file, output_file)


In [15]:
process_missing_sensors()

Processando: ../data/bronze/filtered_parquet/002/task_3.parquet
Processando: ../data/bronze/filtered_parquet/002/task_1.parquet
Processando: ../data/bronze/filtered_parquet/002/task_4.parquet
Processando: ../data/bronze/filtered_parquet/002/task_2.parquet
Processando: ../data/bronze/filtered_parquet/001/task_3.parquet
Processando: ../data/bronze/filtered_parquet/001/task_1.parquet
Processando: ../data/bronze/filtered_parquet/001/task_4.parquet
Processando: ../data/bronze/filtered_parquet/001/task_2.parquet
Processando: ../data/bronze/filtered_parquet/011/task_3.parquet
Processando: ../data/bronze/filtered_parquet/011/task_1.parquet
Processando: ../data/bronze/filtered_parquet/011/task_4.parquet
Processando: ../data/bronze/filtered_parquet/011/task_2.parquet
Processando: ../data/bronze/filtered_parquet/007/task_3.parquet
Processando: ../data/bronze/filtered_parquet/007/task_1.parquet
Processando: ../data/bronze/filtered_parquet/007/task_4.parquet
Processando: ../data/bronze/filtered_par

In [16]:
path1 = "../data/bronze/filtered_parquet/008/OFF_2/task_2.parquet"
path2 = "../data/bronze/filtered_parquet/009/task_6.parquet"

df1 = pd.read_parquet(path1)
df2 = pd.read_parquet(path2)

df1.columns == df2.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True])

In [17]:
df1[acc_tags].info()

<class 'pandas.DataFrame'>
RangeIndex: 187501 entries, 0 to 187500
Data columns (total 28 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   LShank-ACCX    187501 non-null  float32
 1   LShank-ACCY    187501 non-null  float32
 2   LShank-ACCZ    187501 non-null  float32
 3   LShank-GYRO-X  187501 non-null  float32
 4   LShank-GYRO-Y  187501 non-null  float32
 5   LShank-GYRO-Z  187501 non-null  float32
 6   LShank-NC/SC   187501 non-null  float32
 7   RShank-ACCX    0 non-null       float64
 8   RShank-ACCY    0 non-null       float64
 9   RShank-ACCZ    0 non-null       float64
 10  RShank-GYRO-X  0 non-null       float64
 11  RShank-GYRO-Y  0 non-null       float64
 12  RShank-GYRO-Z  0 non-null       float64
 13  RShank-NC/SC   0 non-null       float64
 14  Waist-ACCX     0 non-null       float64
 15  Waist-ACCY     0 non-null       float64
 16  Waist-ACCZ     0 non-null       float64
 17  Waist-GYRO-X   0 non-null       float64


# Análise Estatística e Plots dos Dados Filtrados

Vamos começar implementando as funções para obter 3 tipos de gráficos de acordo com as medições de cada paciente

## Imports e carregamento

In [18]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = "../data/bronze/filtered_parquet/"

Vamos pegar as tasks de cada paciente:

In [19]:
def get_patient_tasks(patient_id):
    patient_path = os.path.join(BASE_PATH, patient_id)

    all_files = []

    # tratamento especial automático do paciente 008
    if patient_id == "008":
        for session in ["OFF_1", "OFF_2"]:
            session_path = os.path.join(patient_path, session)

            if not os.path.exists(session_path):
                continue

            files = [
                os.path.join(session_path, f)
                for f in sorted(os.listdir(session_path))
                if f.endswith(".parquet")
            ]

            all_files.extend(files)

    else:
        files = [
            os.path.join(patient_path, f)
            for f in sorted(os.listdir(patient_path))
            if f.endswith(".parquet")
        ]
        all_files.extend(files)

    return sorted(all_files)

def get_sensor_columns(df):
    ignore = ["timestamp", "patient_id", "task_id", "session", "label"]
    return [col for col in df.columns if col not in ignore]



Função sinal no tempo:

In [20]:
def plot_time_all_channels(patient_id, n_samples=2000):
    files = get_patient_tasks(patient_id)[:4]

    df_sample = pd.read_parquet(files[0])
    channels = get_sensor_columns(df_sample)

    for channel in channels:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes = axes.flatten()

        for i, file in enumerate(files):
            df = pd.read_parquet(file)

            axes[i].plot(df[channel].values[:n_samples])
            axes[i].set_title(os.path.basename(file))
            axes[i].set_xlabel("Tempo")
            axes[i].set_ylabel("Amplitude")

        plt.suptitle(f"Paciente {patient_id} - Sinal - {channel}")
        plt.tight_layout()
        plt.show()

Histogramas:

In [21]:
def plot_hist_all_channels(patient_id):
    files = get_patient_tasks(patient_id)[:4]

    df_sample = pd.read_parquet(files[0])
    channels = get_sensor_columns(df_sample)

    for channel in channels:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes = axes.flatten()

        for i, file in enumerate(files):
            df = pd.read_parquet(file)

            axes[i].hist(df[channel].dropna(), bins=50)
            axes[i].set_title(os.path.basename(file))

        plt.suptitle(f"Paciente {patient_id} - Histograma - {channel}")
        plt.tight_layout()
        plt.show()

Boxplots:

In [22]:
def plot_box_all_channels(patient_id):
    files = get_patient_tasks(patient_id)[:4]

    df_sample = pd.read_parquet(files[0])
    channels = get_sensor_columns(df_sample)

    for channel in channels:
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        axes = axes.flatten()

        for i, file in enumerate(files):
            df = pd.read_parquet(file)

            axes[i].boxplot(df[channel].dropna())
            axes[i].set_title(os.path.basename(file))

        plt.suptitle(f"Paciente {patient_id} - Boxplot - {channel}")
        plt.tight_layout()
        plt.show()

Heatmaps:

In [ ]:
def plot_correlation_heatmaps(patient_id, method="pearson", max_samples=None):
    files = get_patient_tasks(patient_id)[:4]

    if len(files) == 0:
        print("Nenhum arquivo encontrado.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    axes = axes.flatten()

    for i, file in enumerate(files):
        df = pd.read_parquet(file)

        channels = get_sensor_columns(df)

        data = df[channels]

        # opcional: limitar número de amostras (performance)
        if max_samples:
            data = data.iloc[:max_samples]

        # correlação
        corr = data.corr(method=method)

        im = axes[i].imshow(corr.values, aspect='auto')

        axes[i].set_title(os.path.basename(file))

        # labels (pode ficar pesado se tiver muitos canais)
        axes[i].set_xticks(range(len(channels)))
        axes[i].set_yticks(range(len(channels)))

        axes[i].set_xticklabels(channels, rotation=90, fontsize=6)
        axes[i].set_yticklabels(channels, fontsize=6)

    fig.suptitle(f"Paciente {patient_id} - Correlação entre Canais")

    fig.colorbar(im, ax=axes, fraction=0.02)

    plt.tight_layout()
    plt.show()

Vamos agora gerar gráficos para o paciente 001, como temos 11 pacientes (12 se contar o 008 duas vezes) teríamos cerca de 60x4x3 gráficos para cada pessoa:

In [ ]:
plot_time_all_channels("001")
plot_hist_all_channels("001")
plot_box_all_channels("001")
plot_correlation_heatmaps("001", max_samples=2000)

TypeError: plot_time_all_channels() got an unexpected keyword argument 'max_samples'